In [ ]:
# Copyright 2026 Jair Lemmens JairLemmens@gmail.com

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

# http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
import numpy as np
from typing import Literal

In [ ]:

class Frame:
    def __init__(self, origin=np.array([0.0,0.0,0.0]), x=np.array([1.0,0.0,0.0]), y=np.array([0.0,1.0,0.0])):
        self.origin = origin
        self.x = x / np.linalg.norm(x)
        self.y = y - np.dot(y, self.x) * self.x
        self.y /= np.linalg.norm(self.y)
    @property
    def z(self):
        return np.cross(self.x, self.y)
    @property
    def R(self):
        return np.column_stack([self.x, self.y, self.z])

def project_onto_plane(v, normal:np.array=np.array([0.0,0.0,1.0])):
    n = normal / np.linalg.norm(normal)
    return v - np.dot(v, n) * n

def project_to_frame(p, frame):
    return frame.R.T @ (p - frame.origin)

def from_frame(p, frame):
    return frame.origin + frame.R @ p

def ccw_angle(v, normal=np.array([0.0, 0.0, 1.0])):
    vp = project_onto_plane(v,normal)
    ref = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(ref, normal)) > 0.99:
        ref = np.array([0.0, 1.0, 0.0])
    ref -= np.dot(ref, normal) * normal
    ref /= np.linalg.norm(ref)
    return np.arctan2(np.dot(normal, np.cross(ref, vp)),np.dot(ref, vp)) % (2 * np.pi)

def ray_intersection(p1, d1, p2, d2):
    try:
        A = np.column_stack((d1, -d2))
        b = p2 - p1
        t1, t2 = np.linalg.solve(A, b)
        return(p1 + t1 * d1)
    except:
        return(p1)

def joint_at_start(wall,joint):
    return(np.linalg.norm(wall.origin-joint)<1e-6)

def miter_joint(layer0,layer1,frame:Frame = None):
    if frame==None:
        frame=Frame()
    p0 = layer0.wall.end
    p1 = layer0.wall.start
    p2 = layer1.wall.start
    p3 = layer1.wall.end

    if np.linalg.norm(p0 - p2) < 1e-6:
        joint = p0
    elif np.linalg.norm(p0 - p3) < 1e-6:
        joint = p0
    elif np.linalg.norm(p1 - p2) < 1e-6:
        joint = p1
    elif np.linalg.norm(p1 - p3) < 1e-6:
        joint = p1    
        
    d1 = layer0.wall.frame.x
    d2 = layer1.wall.frame.x

    p1_1_3d = layer0.origin + layer0.wall.frame.y * layer0.offset
    p2_1_3d = layer1.origin + layer1.wall.frame.y * layer1.offset

    p1_2_3d = layer0.origin + layer0.wall.frame.y * (layer0.offset + layer0.thickness)
    p2_2_3d = layer1.origin + layer1.wall.frame.y * (layer1.offset + layer1.thickness)
    
    p1_1 = project_to_frame(p1_1_3d, frame)
    p2_1 = project_to_frame(p2_1_3d, frame)

    p1_2 = project_to_frame(p1_2_3d, frame)
    p2_2 = project_to_frame(p2_2_3d, frame)

    d1_plane = frame.R.T @ d1
    d2_plane = frame.R.T @ d2

    one_start = joint_at_start(layer0,joint)
    two_start = joint_at_start(layer1,joint)

    if abs(np.dot(d1_plane, d2_plane)) > 0.999:
        if not one_start and two_start:
            points = layer0.end_points
            start1 = layer1.start_points
            dist = np.linalg.norm(points[0]-start1[0])
            if dist>1e-3:
                points += d1*dist
                layer0.end_points = points
        elif one_start and not two_start:
            points = layer0.start_points
            end1 = layer1.end_points
            dist = np.linalg.norm(points[0]-end1[0])
            if dist>1e-3:
                points -= d1*dist
                layer0.start_points = points
        return

    if one_start and two_start:
        intersection0_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        t0 = np.dot(intersection0_2d - p1_2[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s0 = np.dot(intersection0_2d - p2_1[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])
        t1 = np.dot(intersection1_2d - p1_1[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s1 = np.dot(intersection1_2d - p2_2[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])

        layer0_i0 = p1_2_3d + t0 * d1
        layer1_i0 = p2_1_3d + s0 * d2
        layer0_i1 = p1_1_3d + t1 * d1
        layer1_i1 = p2_2_3d + s1 * d2

        layer0.start_points = np.stack([layer0_i1, layer0_i0])
        layer1.start_points = np.stack([layer1_i0, layer1_i1])

    elif not one_start and not two_start:
        intersection0_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        t0 = np.dot(intersection0_2d - p1_2[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s0 = np.dot(intersection0_2d - p2_1[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])
        t1 = np.dot(intersection1_2d - p1_1[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s1 = np.dot(intersection1_2d - p2_2[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])

        layer0_i0 = p1_2_3d + t0 * d1
        layer1_i0 = p2_1_3d + s0 * d2
        layer0_i1 = p1_1_3d + t1 * d1
        layer1_i1 = p2_2_3d + s1 * d2

        layer0.end_points = np.stack([layer0_i1, layer0_i0])
        layer1.end_points = np.stack([layer1_i0, layer1_i1])

    elif one_start and not two_start:
        intersection0_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        t0 = np.dot(intersection0_2d - p1_1[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s0 = np.dot(intersection0_2d - p2_1[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])
        t1 = np.dot(intersection1_2d - p1_2[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s1 = np.dot(intersection1_2d - p2_2[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])

        layer0_i0 = p1_1_3d + t0 * d1
        layer1_i0 = p2_1_3d + s0 * d2
        layer0_i1 = p1_2_3d + t1 * d1
        layer1_i1 = p2_2_3d + s1 * d2

        layer0.start_points = np.stack([layer0_i0, layer0_i1])
        layer1.end_points = np.stack([layer1_i0, layer1_i1])

    elif not one_start and two_start:
        intersection0_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        t0 = np.dot(intersection0_2d - p1_1[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s0 = np.dot(intersection0_2d - p2_1[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])
        t1 = np.dot(intersection1_2d - p1_2[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s1 = np.dot(intersection1_2d - p2_2[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])

        layer0_i0 = p1_1_3d + t0 * d1
        layer1_i0 = p2_1_3d + s0 * d2
        layer0_i1 = p1_2_3d + t1 * d1
        layer1_i1 = p2_2_3d + s1 * d2

        layer0.end_points = np.stack([layer0_i0, layer0_i1])
        layer1.start_points = np.stack([layer1_i0, layer1_i1])

def butt_joint(layer0, layer1, frame : Frame = None):
    if frame==None:
        frame=Frame()
    p0 = layer0.wall.end
    p1 = layer0.wall.start
    p2 = layer1.wall.start
    p3 = layer1.wall.end

    if np.linalg.norm(p0 - p2) < 1e-6:
        joint = p0
    elif np.linalg.norm(p0 - p3) < 1e-6:
        joint = p0
    elif np.linalg.norm(p1 - p2) < 1e-6:
        joint = p1
    elif np.linalg.norm(p1 - p3) < 1e-6:
        joint = p1    

    d1 = layer0.wall.frame.x
    d2 = layer1.wall.frame.x

    p1_1_3d = layer0.origin + layer0.wall.frame.y * layer0.offset
    p2_1_3d = layer1.origin + layer1.wall.frame.y * layer1.offset

    p1_2_3d = layer0.origin + layer0.wall.frame.y * (layer0.offset + layer0.thickness)
    p2_2_3d = layer1.origin + layer1.wall.frame.y * (layer1.offset + layer1.thickness)

    p1_1 = project_to_frame(p1_1_3d, frame)
    p2_1 = project_to_frame(p2_1_3d, frame)
    p1_2 = project_to_frame(p1_2_3d, frame)
    p2_2 = project_to_frame(p2_2_3d, frame)

    d1_plane = frame.R.T @ d1
    d2_plane = frame.R.T @ d2

    one_start = joint_at_start(layer0, joint)
    two_start = joint_at_start(layer1, joint)

    if abs(np.dot(d1_plane, d2_plane)) > 0.999:
        if not one_start and two_start:
            points = layer0.end_points
            start1 = layer1.start_points
            dist = np.linalg.norm(points[0] - start1[0])
            if dist > 1e-3:
                points += d1 * dist
                layer0.end_points = points
        elif one_start and not two_start:
            points = layer0.start_points
            end1 = layer1.end_points
            dist = np.linalg.norm(points[0] - end1[0])
            if dist > 1e-3:
                points -= d1 * dist
                layer0.start_points = points
        return

    def reconstruct(i2d, p1_3d, p2_3d):
        p1 = project_to_frame(p1_3d, frame)
        p2 = project_to_frame(p2_3d, frame)

        t = np.dot(i2d - p1[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s = np.dot(i2d - p2[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])

        return p1_3d + t * d1, p2_3d + s * d2
    if one_start and two_start:
        intersection0_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])

        layer0_i0, _ = reconstruct(intersection0_2d, p1_1_3d, p2_1_3d)
        layer0_i1, _ = reconstruct(intersection1_2d, p1_2_3d, p2_1_3d)

        layer0.start_points = np.stack([layer0_i0, layer0_i1])

    elif one_start and not two_start:
        intersection0_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        layer0_i0, _ = reconstruct(intersection0_2d, p1_1_3d, p2_2_3d)
        layer0_i1, _ = reconstruct(intersection1_2d, p1_2_3d, p2_2_3d)

        layer0.start_points = np.stack([layer0_i0, layer0_i1])

    elif not one_start and two_start:
        intersection0_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        layer0_i0, _ = reconstruct(intersection0_2d, p1_1_3d, p2_2_3d)
        layer0_i1, _ = reconstruct(intersection1_2d, p1_2_3d, p2_2_3d)

        layer0.end_points = np.stack([layer0_i0, layer0_i1])

    else:
        intersection0_2d = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        intersection1_2d = ray_intersection(p1_2[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])

        layer0_i0, _ = reconstruct(intersection0_2d, p1_1_3d, p2_1_3d)
        layer0_i1, _ = reconstruct(intersection1_2d, p1_2_3d, p2_1_3d)

        layer0.end_points = np.stack([layer0_i0, layer0_i1])

def double_butt_joint(layer0, layer1, frame: Frame = None):
    if frame==None:
        frame=Frame()
    p0 = layer0.wall.end
    p1 = layer0.wall.start
    p2 = layer1.wall.start
    p3 = layer1.wall.end

    if np.linalg.norm(p0 - p2) < 1e-6:
        joint = p0
    elif np.linalg.norm(p0 - p3) < 1e-6:
        joint = p0
    elif np.linalg.norm(p1 - p2) < 1e-6:
        joint = p1
    elif np.linalg.norm(p1 - p3) < 1e-6:
        joint = p1    

    d1 = layer0.wall.frame.x
    d2 = layer1.wall.frame.x

    p1_1_3d = layer0.origin + layer0.wall.frame.y * layer0.offset
    p2_1_3d = layer1.origin + layer1.wall.frame.y * layer1.offset
    p1_2_3d = layer0.origin + layer0.wall.frame.y * (layer0.offset + layer0.thickness)
    p2_2_3d = layer1.origin + layer1.wall.frame.y * (layer1.offset + layer1.thickness)

    p1_1 = project_to_frame(p1_1_3d, frame)
    p2_1 = project_to_frame(p2_1_3d, frame)
    p1_2 = project_to_frame(p1_2_3d, frame)
    p2_2 = project_to_frame(p2_2_3d, frame)

    d1_plane = frame.R.T @ d1
    d2_plane = frame.R.T @ d2

    one_start = joint_at_start(layer0, joint)
    two_start = joint_at_start(layer1, joint)

    if abs(np.dot(d1_plane, d2_plane)) > 0.999:
        if not one_start and two_start:
            points = layer0.end_points
            start1 = layer1.start_points
            dist = np.linalg.norm(points[0] - start1[0])
            if dist > 1e-3:
                points += d1 * dist
                layer0.end_points = points
        elif one_start and not two_start:
            points = layer0.start_points
            end1 = layer1.end_points
            dist = np.linalg.norm(points[0] - end1[0])
            if dist > 1e-3:
                points -= d1 * dist
                layer0.start_points = points
        return

    def reconstruct(i2d, p1_3d, p2_3d):
        p1 = project_to_frame(p1_3d, frame)
        p2 = project_to_frame(p2_3d, frame)

        t = np.dot(i2d - p1[:2], d1_plane[:2]) / np.dot(d1_plane[:2], d1_plane[:2])
        s = np.dot(i2d - p2[:2], d2_plane[:2]) / np.dot(d2_plane[:2], d2_plane[:2])

        return p1_3d + t * d1, p2_3d + s * d2

    if one_start and two_start:
        i0 = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        i1 = ray_intersection(p1_2[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        i2 = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        intersection0, _ = reconstruct(i0, p1_1_3d, p2_1_3d)
        intersection1, _ = reconstruct(i1, p1_2_3d, p2_1_3d)
        _, intersection2 = reconstruct(i2, p1_1_3d, p2_2_3d)

        layer0.start_points = np.stack([intersection0, intersection1])
        layer1.start_points = np.stack([intersection0, intersection2])

    elif one_start and not two_start:
        i0 = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])
        i1 = ray_intersection(p1_2[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])
        i2 = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])

        intersection0, _ = reconstruct(i0, p1_1_3d, p2_2_3d)
        intersection1, _ = reconstruct(i1, p1_2_3d, p2_2_3d)
        _, intersection2 = reconstruct(i2, p1_1_3d, p2_1_3d)

        layer0.start_points = np.stack([intersection0, intersection1])
        layer1.end_points = np.stack([intersection2, intersection0])

    elif not one_start and two_start:
        i0 = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])
        i1 = ray_intersection(p1_2[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])
        i2 = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])

        intersection0, _ = reconstruct(i0, p1_1_3d, p2_2_3d)
        intersection1, _ = reconstruct(i1, p1_2_3d, p2_2_3d)
        _, intersection2 = reconstruct(i2, p1_1_3d, p2_1_3d)

        layer0.end_points = np.stack([intersection0, intersection1])
        layer1.start_points = np.stack([intersection2, intersection0])

    else:
        i0 = ray_intersection(p1_1[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        i1 = ray_intersection(p1_2[:2], d1_plane[:2],p2_1[:2], d2_plane[:2])
        i2 = ray_intersection(p1_1[:2], d1_plane[:2],p2_2[:2], d2_plane[:2])

        intersection0, _ = reconstruct(i0, p1_1_3d, p2_1_3d)
        intersection1, _ = reconstruct(i1, p1_2_3d, p2_1_3d)
        _, intersection2 = reconstruct(i2, p1_1_3d, p2_2_3d)

        layer0.end_points = np.stack([intersection0, intersection1])
        layer1.end_points = np.stack([intersection0, intersection2])

def solve_joint(walls):
    if len(walls)<2:
        return

    if np.linalg.norm(walls[0].start-walls[1].start)<1e-5 or np.linalg.norm(walls[0].start-walls[1].end)<1e-5:
        joint = walls[0].start
    else:
        joint = walls[0].end

    dirs = []
    flipped = []
    for wall in walls:
        if joint_at_start(wall, joint):
            v = -wall.axis
            flipped.append(True)
        else:
            v = wall.axis
            flipped.append(False)
        dirs.append(v)

    ccw = np.argsort([ccw_angle(v) for v in dirs])

    walls = [walls[i] for i in ccw]
    flipped = [flipped[i] for i in ccw]
    num_walls = len(walls)

    ###map priorities
    priorities = {}
    priorities_indices = {}
    for n,wall in enumerate(walls):
        layer_priorities = []
        for layer in wall.layers:
            layer_priorities.append(layer.priority)
            if layer.priority in priorities:
                priorities[layer.priority].append(layer)
                priorities_indices[layer.priority].append(n)
            else:
                priorities[layer.priority] = [layer]
                priorities_indices[layer.priority] = [n]

    layer_joints = {}
    for priority in reversed(sorted(priorities.keys())):
        layers = priorities[priority]
        indices = priorities_indices[priority]

        for index,layer in zip(indices,layers):
            candidate_index = index
            ###max number of connection per joint
            for n in range(4):
                
                if layer in layer_joints.values() or layer in layer_joints.keys():
                    break

                scan_cw = False
                wall_layer_index = layer.wall.layers.index(layer)
                if len(layer.wall.priorities[wall_layer_index:])>0:
                    if sorted(layer.wall.priorities[wall_layer_index:])[-1] > priority:
                        scan_cw = True

                if (flipped[index] or scan_cw) and not (flipped[index] and scan_cw):
                    #go clockwise
                    candidate_index = candidate_index - 1
                else:
                    #go counterclockwise
                    candidate_index =candidate_index + 1

                candidate_index = (candidate_index)%num_walls
                candidate_wall = walls[candidate_index]                

                flip_layers = scan_cw
                if not flipped[index] and flipped[candidate_index]:
                    flip_layers = not scan_cw
                if flipped[index] and not flipped[candidate_index]:
                    flip_layers = not scan_cw
        
                candidate_priorities = (reversed(candidate_wall.priorities) if flip_layers else candidate_wall.priorities)
                candidate_layers = (reversed(candidate_wall.layers) if flip_layers else candidate_wall.layers)

                for candidate_layer, candidate_priority in zip(candidate_layers, candidate_priorities):
                    if candidate_priority > priority:
                        butt_joint(layer,candidate_layer)
                        layer_joints[layer] = candidate_layer
                        #print(f'higher_priority butt joint {index} {layer.wall.layers.index(layer)},{candidate_index} {candidate_layer.wall.layers.index(candidate_layer)}')
                        break
                    elif candidate_priority == priority:
                        if candidate_layer in layer_joints.values() or candidate_layer in layer_joints.keys():
                            butt_joint(layer,candidate_layer)
                            layer_joints[layer] = candidate_layer
                            #print(f'visited butt joint {index} {layer.wall.layers.index(layer)},{candidate_index} {candidate_layer.wall.layers.index(candidate_layer)}')
                            break
                        else:
                            if layer.joint_style == 'miter_joint':
                                miter_joint(layer,candidate_layer)
                            else:
                                double_butt_joint(layer,candidate_layer)
                            layer_joints[layer] = candidate_layer
                            #print(f'miter joint {index} {layer.wall.layers.index(layer)},{candidate_index} {candidate_layer.wall.layers.index(candidate_layer)} ')
                            break
    return(layer_joints)


class WallLayer:
    def __init__(self,wall,offset,thickness,priority,name='unnamed',joint_style: Literal["miter_joint","butt_joint"]="miter_joint"):
        self.wall = wall
        self.offset = offset
        self.thickness = thickness
        self.priority = priority
        self.joint_style = joint_style
        self.name = name
        self.start_points = np.array([self.wall.start + self.wall.frame.y*self.offset,self.wall.start + self.wall.frame.y*(self.offset+self.thickness)]) 
        self.end_points = np.array([self.wall.end + self.wall.frame.y*self.offset,self.wall.end + self.wall.frame.y*(self.offset+self.thickness)])
        
    @property
    def axis(self):
        return(self.wall.axis)
    @property
    def origin(self):
        return(self.wall.start)

class Wall:
    def __init__(self,start:np.array,end:np.array,layers:[WallLayer]=None,name = 'unnamed',height:float=3,z:np.array=np.array([0.0,0.0,1.0]),face=None):
        self.start = start
        self.end = end
        self.layers = [] if layers == None else layers
        self.name = name
        self.frame = Frame(self.start,self.end-self.start,np.cross(self.end-self.start,z))
    
    @property
    def axis(self):
        return(self.frame.x)
    @property
    def origin(self):
        return(self.frame.origin)
    @property
    def priorities(self):
        return([layer.priority for layer in self.layers])


In [ ]:

wall0 = Wall(np.array([0.0,0.0,0.0]),np.array([3.0,1.0,0.0]))
wall0.layers.append(WallLayer(wall0,0,.2,100,'concrete'))
wall0.layers.append(WallLayer(wall0,.2,.3,80,'insulation','miter_joint'))

wall1 = Wall(np.array([-3.0,1.0,0.0]),np.array([0.0,0.0,0.0]))
wall1.layers.append(WallLayer(wall1,0,.2,100,'concrete'))
wall1.layers.append(WallLayer(wall1,.2,.3,80,'insulation',joint_style='miter_joint'))

wall2 = Wall(np.array([1.0,-3.0,0.0]),np.array([0.0,0.0,0.0]))
wall2.layers.append(WallLayer(wall2,0,.2,100,'concrete'))

solve_joint([wall0,wall1,wall2])